# Generate Process Descriptions from Role-Specific OCEL Files

This notebook:
1. Loads each role-specific OCEL file from `role_logs/`
2. Extracts process structure (activities, objects, flows)
3. Uses GPT-5.4-mini to generate comprehensive markdown descriptions
4. Saves descriptions as markdown files in `role_logs/`

In [ ]:
import os
import json
import pm4py
from openai import OpenAI
from collections import Counter, defaultdict
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
# Configuration
LOG_DIR = "role_logs"
OUTPUT_SUFFIX = "_process_description.md"
OPENAI_MODEL = "gpt-5.4-mini"

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Find all role-specific OCEL files
role_files = sorted([
    f for f in os.listdir(LOG_DIR)
    if f.startswith("ocel_") and f.endswith(".json")
])

print(f"Found {len(role_files)} role-specific OCEL files:")
for f in role_files:
    print(f"  - {f}")

In [ ]:
def extract_process_info(ocel_path):
    """
    Extract comprehensive process information from OCEL file.
    Returns a structured summary for LLM analysis.
    """
    with open(ocel_path) as f:
        data = json.load(f)
    
    # Debug: Print top-level keys to understand structure
    print(f"  DEBUG: Top-level keys in JSON: {list(data.keys())}")
    
    # Load with pm4py for additional analysis
    ocel = pm4py.read_ocel2_json(ocel_path)
    
    # Extract basic statistics - These files use simple arrays, not OCEL 2.0 format
    events = data.get("events", [])
    objects = data.get("objects", [])
    
    # Activity analysis - These files use "type" field for activity name
    activities = [e.get("type") for e in events if e.get("type")]
    activity_counts = Counter(activities)
    
    # Object type analysis
    object_types = defaultdict(list)
    for obj in objects:
        obj_type = obj.get("type")
        if obj_type:
            object_types[obj_type].append(obj.get("id"))
    
    # Event-object relationships
    event_object_pairs = []
    # Build object type lookup
    obj_type_map = {obj.get("id"): obj.get("type") for obj in objects}
    
    for event in events:
        activity = event.get("type")
        relationships = event.get("relationships", [])
        
        for rel in relationships:
            obj_id = rel.get("objectId")
            qualifier = rel.get("qualifier", "relates_to")
            obj_type = obj_type_map.get(obj_id)
            
            if obj_type and activity:
                event_object_pairs.append((activity, qualifier, obj_type))
    
    # Activity flow patterns (simple directly-follows)
    activity_flows = []
    sorted_events = sorted(events, key=lambda e: e.get("time", ""))
    for i in range(len(sorted_events) - 1):
        act1 = sorted_events[i].get("type")
        act2 = sorted_events[i + 1].get("type")
        if act1 and act2:
            activity_flows.append((act1, act2))
    
    flow_counts = Counter(activity_flows)
    top_flows = flow_counts.most_common(10)
    
    # Object interaction patterns
    object_interaction_counts = Counter(event_object_pairs)
    top_interactions = object_interaction_counts.most_common(15)
    
    return {
        "total_events": len(events),
        "total_objects": len(objects),
        "activities": dict(activity_counts),
        "object_types": {k: len(v) for k, v in object_types.items()},
        "top_flows": top_flows,
        "top_interactions": top_interactions,
        "unique_activities": len(activity_counts),
        "unique_object_types": len(object_types)
    }

In [ ]:
def generate_process_description(role_name, process_info):
    """
    Generate comprehensive process description using GPT-5.4-mini.
    """
    
    # Format process information for the prompt
    activities_str = "\n".join([
        f"  - {act}: {count} occurrences"
        for act, count in sorted(process_info["activities"].items(), 
                                  key=lambda x: x[1], reverse=True)
    ])
    
    object_types_str = "\n".join([
        f"  - {obj_type}: {count} objects"
        for obj_type, count in process_info["object_types"].items()
    ])
    
    flows_str = "\n".join([
        f"  - {flow[0]} → {flow[1]}: {count} times"
        for flow, count in process_info["top_flows"]
    ])
    
    interactions_str = "\n".join([
        f"  - Activity '{interaction[0]}' {interaction[1]} {interaction[2]}: {count} times"
        for interaction, count in process_info["top_interactions"]
    ])
    
    prompt = f"""You are a business process expert specializing in object-centric process mining. 
Analyze this object-centric event log for the '{role_name}' role and provide a comprehensive 
textual description following this EXACT structure:

## OBJECT-CENTRIC PROCESS DESCRIPTION: {role_name.upper()}

[Opening paragraph describing the overall process and the role's purpose]

---

## 1. Process Overview

[Detailed overview of what this role does in the software development process, including key responsibilities]

---

## 2. Main Activities

[Describe each activity and its purpose, grouping related activities together]

---

## 3. Object Interactions

[Describe how this role interacts with different object types (commits, issues, tasks, users) and what these interactions mean]

---

## 4. Process Flow Patterns

[Describe the typical flow of activities, including common sequences and patterns]

---

## 5. Key Characteristics

[Highlight distinctive features of this role's process behavior]

---

## Concise Business Interpretation

[Summary paragraph explaining the role's contribution to the overall software development lifecycle]

---

## Process Statistics

- **Total Events**: {process_info['total_events']}
- **Total Objects**: {process_info['total_objects']}
- **Unique Activities**: {process_info['unique_activities']}
- **Object Types**: {process_info['unique_object_types']}

### Activities (by frequency):
{activities_str}

### Object Types:
{object_types_str}

### Top Activity Flows:
{flows_str}

### Top Object Interactions:
{interactions_str}

Write in clear, professional language suitable for business stakeholders and process analysts. 
Follow the structure exactly as shown above. Focus on the object-centric nature of the process, 
explaining how activities relate to multiple objects simultaneously."""
    
    print(f"\nGenerating description for {role_name}...")
    
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    
    return response.choices[0].message.content

In [ ]:
# Process each role-specific OCEL file
print("="*70)
print("PROCESSING ROLE-SPECIFIC OCEL FILES")
print("="*70)

for role_file in role_files:
    # Extract role name from filename (e.g., "ocel_maintainer.json" -> "maintainer")
    role_name = role_file.replace("ocel_", "").replace(".json", "")
    
    print(f"\n{'='*70}")
    print(f"Processing: {role_name}")
    print(f"{'='*70}")
    
    # Full path to OCEL file
    ocel_path = os.path.join(LOG_DIR, role_file)
    
    # Extract process information
    print(f"Extracting process information from {role_file}...")
    process_info = extract_process_info(ocel_path)
    
    print(f"  Events: {process_info['total_events']}")
    print(f"  Objects: {process_info['total_objects']}")
    print(f"  Activities: {process_info['unique_activities']}")
    print(f"  Object Types: {process_info['unique_object_types']}")
    
    # Generate description using LLM
    description = generate_process_description(role_name, process_info)
    
    # Save to markdown file
    output_file = os.path.join(LOG_DIR, f"{role_name}{OUTPUT_SUFFIX}")
    with open(output_file, 'w') as f:
        f.write(description)
    
    print(f"✓ Saved description to: {output_file}")
    print(f"\nPreview (first 500 characters):")
    print("-" * 70)
    print(description[:500] + "...")
    print("-" * 70)

print(f"\n{'='*70}")
print("COMPLETED: All role descriptions generated")
print(f"{'='*70}")
print(f"\nGenerated {len(role_files)} markdown files in {LOG_DIR}/")

In [ ]:
# List all generated markdown files
print("\nGenerated Process Description Files:")
print("="*70)

md_files = sorted([
    f for f in os.listdir(LOG_DIR)
    if f.endswith(OUTPUT_SUFFIX)
])

for md_file in md_files:
    file_path = os.path.join(LOG_DIR, md_file)
    file_size = os.path.getsize(file_path)
    print(f"  ✓ {md_file:<50} ({file_size:,} bytes)")

print(f"\nTotal: {len(md_files)} markdown files")